# 04 — Gold: бизнес-витрины
Из чистого silver строим денормализованные витрины с готовыми метриками:1. Выручка по сегментам (join + агрегация)2. Топ-клиенты по городам (оконная функция)3. Динамика транзакций по днямЗапускать после `02_silver`. Витрины пишем в `s3a://gold/` как **чистый Parquet** (не Delta): витрины перезаписываются целиком, история/MERGE не нужны — это нужно на silver. Простой Parquet идеально читается ClickHouse.

## 1. Spark-сессия и чтение silver

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("gold").getOrCreate()

clients = spark.read.format("delta").load("s3a://silver/clients")
accounts = spark.read.format("delta").load("s3a://silver/accounts")
transactions = spark.read.format("delta").load("s3a://silver/transactions")

print("clients:", clients.count(), "| accounts:", accounts.count(),
      "| tx:", transactions.count())

clients: 9828 | accounts: 14171 | tx: 189879


## 2. Соединяем факт с измерениями
Цепочка: транзакция → счёт (по `account_id`) → клиент (по `client_id`).Получаем широкую таблицу, где у каждой транзакции есть сегмент и город клиента.

In [2]:
joined = (transactions
    .join(accounts.select("account_id", "client_id"), "account_id")
    .join(clients.select("client_id", "full_name", "city", "segment"), "client_id"))

# только успешные операции участвуют в выручке
completed = joined.filter(F.col("status") == "completed")
print("Успешных транзакций после join:", completed.count())

Успешных транзакций после join: 105904


## 3. Витрина 1: выручка по сегментам
Группируем по сегменту: сумма выручки, число операций, средний чек.

In [3]:
revenue_by_segment = (completed
    .groupBy("segment")
    .agg(F.round(F.sum("amount"), 2).alias("total_revenue"),
         F.count("*").alias("tx_count"),
         F.round(F.avg("amount"), 2).alias("avg_amount"))
    .orderBy(F.col("total_revenue").desc()))

(revenue_by_segment.write.mode("overwrite")
    .parquet("s3a://gold/revenue_by_segment"))

revenue_by_segment.show(truncate=False)

+--------+---------------+--------+----------+
|segment |total_revenue  |tx_count|avg_amount|
+--------+---------------+--------+----------+
|mass    |1.58187843711E9|63222   |25021.01  |
|affluent|5.3854132682E8 |21662   |24861.11  |
|private |5.2129350162E8 |21020   |24799.88  |
+--------+---------------+--------+----------+



## 4. Витрина 2: топ-5 клиентов по городам
Оконная функция `row_number()` ранжирует клиентов внутри каждого городапо убыванию оборота. Берём топ-5 на город.

In [4]:
client_revenue = (completed
    .groupBy("client_id", "full_name", "city")
    .agg(F.round(F.sum("amount"), 2).alias("revenue")))

w = Window.partitionBy("city").orderBy(F.col("revenue").desc())
top_clients = (client_revenue
    .withColumn("rank_in_city", F.row_number().over(w))
    .filter(F.col("rank_in_city") <= 5))

(top_clients.write.mode("overwrite")
    .parquet("s3a://gold/top_clients"))

# покажем топ для одного города
top_clients.filter(F.col("city") == "Москва").orderBy("rank_in_city").show(truncate=False)

+---------+----------------+------+---------+------------+
|client_id|full_name       |city  |revenue  |rank_in_city|
+---------+----------------+------+---------+------------+
|4231     |Анна Иванов     |Москва|761717.16|1           |
|6554     |Наталья Кузнецов|Москва|704941.42|2           |
|5010     |Наталья Петров  |Москва|681726.25|3           |
|8759     |Ольга Морозов   |Москва|681198.56|4           |
|4315     |Дмитрий Смирнов |Москва|680576.77|5           |
+---------+----------------+------+---------+------------+



## 5. Витрина 3: динамика по дням
Группируем по дате: число операций и выручка за день.

In [5]:
daily = (completed
    .withColumn("dt", F.to_date("ts"))
    .groupBy("dt")
    .agg(F.count("*").alias("tx_count"),
         F.round(F.sum("amount"), 2).alias("daily_revenue"))
    .orderBy("dt"))

(daily.write.mode("overwrite")
    .parquet("s3a://gold/daily_transactions"))

daily.show(10, truncate=False)

+----------+--------+---------------+
|dt        |tx_count|daily_revenue  |
+----------+--------+---------------+
|2026-05-23|22068   |5.5033355252E8 |
|2026-05-24|83836   |2.09137971303E9|
+----------+--------+---------------+



## 6. Проверка: все витрины на месте

In [6]:
for name in ["revenue_by_segment", "top_clients", "daily_transactions"]:
    df = spark.read.parquet(f"s3a://gold/{name}")
    print(f"{name:22s} -> {df.count():>5} строк")

revenue_by_segment     ->     3 строк
top_clients            ->    45 строк
daily_transactions     ->     2 строк


## Готово
Три gold-витрины в `s3a://gold/`. Это денормализованные данные с готовымиметриками — их остаётся выгрузить в ClickHouse и показать в Metabase.Дальше — `05_clickhouse`: выгрузка витрин в ClickHouse.